!! POT SER QUE FALLI SI L'ORDINADOR NO TÉ PROU MEMÒRIA RAM

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from utils.tobii import (
    compute_regression_metrics,
    detect_regressions,
    extract_fixations,
    get_hits,
    process_all_texts,
    read_all_data,
    read_data,
    z_score,
)

pd.set_option("display.max_columns", None)

In [ ]:
mused_chosen = pd.read_csv("../data/mused_chosen_data.csv")
mused_chosen["num_id"] = mused_chosen.id.str.split("_").str[1].astype(int)
texts = dict(zip(mused_chosen.num_id, mused_chosen.text_clean))
text_ids = mused_chosen.num_id.tolist()

In [ ]:
PROJECT = "../data/tobii/"
general = (
    pd.read_csv(PROJECT + "general.tsv", sep="\t")
    .drop(["Event", "Recording timestamp", "Computer timestamp"], axis=1)
    .drop_duplicates()
)
# general.head()

In [ ]:
aoi_hit, calibration_dfs, participants, dfs = read_all_data("../data/tobii/all_parquets")
part = list(dict.fromkeys([x[0] for x in participants]))  # unique names in order

In [ ]:
# dfs_part = [pd.concat(dfs[i * 4 : (i + 1) * 4]) for i in range(len(part))]

In [ ]:
# dfs té un df per fitxer de parquet
# text_dfs té un df per persona i per text
text_dfs = {}
aoi_cols_dict = {}
tfds = {}  # Total fixation durations dict user: text_id: tfds
for part_idx in range(len(part)):
    participant = part[part_idx]
    text_dfs_part, aoi_cols_part = process_all_texts(
        dfs[part_idx * 4 : (part_idx + 1) * 4], aoi_hit
    )
    text_dfs[participant] = text_dfs_part
    aoi_cols_dict[participant] = aoi_cols_part

    tfds[participant] = {}
    for text_id in text_ids:
        _df = text_dfs_part[text_id]
        # Total fixation duration
        all_aois = aoi_cols_part[text_id]
        tfd = _df[_df["event_type"] == "Fixation"].groupby(by="AOI")["duration"].sum()
        tfd = tfd.reindex(all_aois, fill_value=0).to_numpy()
        tfds[participant][text_id] = tfd / tfd.sum()

In [ ]:
tfds["karla"]

# MuSeD spans

In [ ]:
data = pd.read_csv("../data/chosen_data_full.csv")
data["num_id"] = data["id"].str.replace("video_", "").astype(int)
# data["label_clean"] =  data["label_clean"].apply(lambda x: eval(x))

In [ ]:
annotations = {}  # max-normalized (for filtering)
annotations_sum = {}  # sum-normalized (for comparison with eye-tracking)
per_label_annotations = {}  # {label: {text_id: sum-normalized array}}

for text_id in data.num_id.tolist():
    row = data[data.num_id == text_id].iloc[0]
    text = row["text_clean"]

    matches = list(re.finditer(r"\S+", text))
    tokens = [m.group() for m in matches]
    starts = [m.start() for m in matches]
    idx2token = dict(zip(starts, range(len(tokens))))

    labels = eval(row["label_clean"])
    _annotations = np.zeros(len(tokens))
    _per_label = {}  # label -> raw count array

    for annotation_id in range(len(labels)):
        start = labels[annotation_id]["start"]
        end = labels[annotation_id]["end"]
        while idx2token.get(start) is None:
            start += 1
        while idx2token.get(end) is None:
            end -= 1
        _annotations[idx2token[start] : idx2token[end] + 1] += 1

        for tag in labels[annotation_id]["labels"]:
            if tag not in _per_label:
                _per_label[tag] = np.zeros(len(tokens))
            _per_label[tag][idx2token[start] : idx2token[end] + 1] += 1

    # max normalization: useful to filter texts where most tokens are annotated
    _annotations_max = np.divide(_annotations.copy(), max(_annotations.max(), 1))
    annotations[text_id] = _annotations_max

    # sum normalization: probabilities for comparison with eye-tracking
    total = _annotations.sum()
    annotations_sum[text_id] = _annotations / max(total, 1)

    # per-label: sum-normalized
    for tag, raw in _per_label.items():
        if tag not in per_label_annotations:
            per_label_annotations[tag] = {}
        per_label_annotations[tag][text_id] = raw / max(raw.sum(), 1)

print(f"{len(per_label_annotations)} label types: {sorted(per_label_annotations.keys())}")

In [ ]:
# {k: (v, v.mean()) for k, v in sorted(annotations.items(), key=lambda item: item[1].mean())}

# Comparison: eye-tracking vs. span annotations

In [ ]:
# Filter out texts where all tokens are annotated (no discriminative signal)
annotated_texts = [
    tid for tid, v in annotations.items() if (v.mean() < 1.0 and v.mean() > 0)
]
print(f"{len(annotated_texts)} texts with partial annotations (out of {len(annotations)})")
print(f"Excluded (all annotated): {sorted(set(annotations) - set(annotated_texts))}")

In [ ]:
EPS = 1e-12  # avoid log(0)


def safe_cross_entropy(p, q):
    """Cross-entropy H(p, q) = -sum(p * log(q)), with epsilon for numerical stability."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    q = np.clip(q, EPS, None)
    return -np.sum(p * np.log(q))


def safe_kl_divergence(p, q):
    """KL(p || q) = sum(p * log(p / q)), with epsilon for numerical stability."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = np.clip(p, EPS, None)
    q = np.clip(q, EPS, None)
    return np.sum(p * np.log(p / q))


def jensen_shannon_divergence(p, q, base2=True):
    """JSD(p || q) = 0.5 * KL(p || m) + 0.5 * KL(q || m), where m = 0.5 * (p + q)."""
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    m = 0.5 * (p + q)
    jsd = 0.5 * safe_kl_divergence(p, m) + 0.5 * safe_kl_divergence(q, m)
    if base2:
        jsd /= np.log(2)
    return jsd


def compute_metrics(eye, ann):
    """Compute Spearman, cross-entropy, and KL-divergence for two aligned vectors."""
    n = min(len(eye), len(ann))
    eye, ann = eye[:n], ann[:n]
    if n == 0:
        return None
    rho, _ = stats.spearmanr(eye, ann)
    return {
        "spearman": rho,
        "cross_entropy": safe_cross_entropy(ann, eye),
        "kl_div": safe_kl_divergence(ann, eye),
        "js_div": jensen_shannon_divergence(ann, eye),
    }


# Compute per-participant metrics (overall)
metrics_records = []

for participant in part:
    for text_id in annotated_texts:
        if text_id not in tfds.get(participant, {}):
            continue
        m = compute_metrics(tfds[participant][text_id], annotations_sum[text_id])
        if m is not None:
            metrics_records.append({"participant": participant, "text_id": text_id, **m})

metrics_df = pd.DataFrame(metrics_records)
print(f"Computed {len(metrics_df)} (participant, text) pairs")

In [ ]:
# Per-participant summary
print("=== Spearman correlation (eye-tracking vs annotations, per participant) ===")
print(metrics_df.groupby("participant")["spearman"].agg(["mean", "std", "count"]))
print()
print("=== Cross-entropy H(annotation | eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["cross_entropy"].agg(["mean", "std", "count"]))
print()
print("=== KL-divergence KL(annotation || eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["kl_div"].agg(["mean", "std", "count"]))
print("=== JS-divergence JS(annotation || eye-tracking, per participant) ===")
print(metrics_df.groupby("participant")["js_div"].agg(["mean", "std", "count"]))

In [ ]:
# Averaged across participants per text
text_avg = metrics_df.groupby("text_id").agg(
    spearman_mean=("spearman", "mean"),
    spearman_std=("spearman", "std"),
    ce_mean=("cross_entropy", "mean"),
    ce_std=("cross_entropy", "std"),
    kl_mean=("kl_div", "mean"),
    kl_std=("kl_div", "std"),
    js_mean=("js_div", "mean"),
    js_std=("js_div", "std"),
    n_participants=("participant", "count"),
)
print("=== Per-text metrics (averaged across participants) ===")
print(text_avg.sort_values("spearman_mean", ascending=False).round(2).to_string())
print()
print(f"Overall mean Spearman rho: {text_avg['spearman_mean'].mean():.2f}")
print(f"Overall mean cross-entropy: {text_avg['ce_mean'].mean():.2f}")
print(f"Overall mean KL-divergence: {text_avg['kl_mean'].mean():.2f}")
print(f"Overall mean JS-divergence: {text_avg['js_mean'].mean():.2f}")

In [ ]:
# Visualization: per text, averaged across participants
gender_map = dict(zip(general["Participant name"], general["sexe"]))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))


axes[0].bar(range(len(text_avg)), text_avg["spearman_mean"])
axes[0].set_xlabel("Text ID")
axes[0].set_ylabel("Mean Spearman rho")
axes[0].set_title("Spearman correlation per text (avg across participants)")
axes[0].axhline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].bar(range(len(text_avg)), text_avg["ce_mean"])
axes[1].set_xlabel("Text ID")
axes[1].set_ylabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per text (avg across participants)")

axes[2].bar(range(len(text_avg)), text_avg["js_mean"])
axes[2].set_xlabel("Text ID")
axes[2].set_ylabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per text (avg across participants)")

plt.tight_layout()
plt.show()

In [ ]:
# Per-participant bar chart
part_summary = metrics_df.groupby("participant").agg(
    spearman_mean=("spearman", "mean"),
    ce_mean=("cross_entropy", "mean"),
    kl_mean=("kl_div", "mean"),
    js_mean=("js_div", "mean"),
)
part_summary["sex"] = part_summary.index.map(gender_map)
part_summary = part_summary.sort_values("spearman_mean", ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ["#1f77b4" if s == "home" else "#e377c2" for s in part_summary["sex"]]

axes[0].barh(part_summary.index, part_summary["spearman_mean"], color=colors)
axes[0].set_xlabel("Mean Spearman rho")
axes[0].set_title("Spearman correlation per participant")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].barh(part_summary.index, part_summary["ce_mean"], color=colors)
axes[1].set_xlabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per participant")

axes[2].barh(part_summary.index, part_summary["js_mean"], color=colors)
axes[2].set_xlabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per participant")

plt.tight_layout()
plt.show()

# Per-label span analysis

In [ ]:
# Compute metrics per label type from span annotations
label_metrics_records = []

for tag, tag_anns in per_label_annotations.items():
    for participant in part:
        for text_id in annotated_texts:
            if text_id not in tfds.get(participant, {}):
                continue
            if text_id not in tag_anns:
                continue
            m = compute_metrics(tfds[participant][text_id], tag_anns[text_id])
            if m is not None:
                label_metrics_records.append(
                    {
                        "label": tag,
                        "participant": participant,
                        "text_id": text_id,
                        **m,
                    }
                )

label_metrics_df = pd.DataFrame(label_metrics_records)
print(
    f"Computed {len(label_metrics_df)} records across {label_metrics_df['label'].nunique()} label types"
)

In [ ]:
# Per-label summary (averaged across participants and texts)
print("=== Metrics per span label type ===")
print(
    label_metrics_df.groupby("label")
    .agg(
        spearman_mean=("spearman", "mean"),
        spearman_std=("spearman", "std"),
        ce_mean=("cross_entropy", "mean"),
        kl_mean=("kl_div", "mean"),
        js_mean=("js_div", "mean"),
        n=("text_id", "count"),
    )
    .sort_values("spearman_mean", ascending=False)
    .round(2)
    .to_string()
)

In [ ]:
# Visualize per-label metrics
label_summary = (
    label_metrics_df.groupby("label")
    .agg(spearman=("spearman", "mean"), ce=("cross_entropy", "mean"), js=("js_div", "mean"))
    .sort_values("spearman", ascending=True)
)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

axes[0].barh(label_summary.index, label_summary["spearman"])
axes[0].set_xlabel("Mean Spearman rho")
axes[0].set_title("Spearman per label type")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5)

axes[1].barh(label_summary.index, label_summary["ce"])
axes[1].set_xlabel("Mean cross-entropy")
axes[1].set_title("Cross-entropy per label type")

axes[2].barh(label_summary.index, label_summary["js"])
axes[2].set_xlabel("Mean JS-divergence")
axes[2].set_title("JS-divergence per label type")

plt.tight_layout()
plt.show()

# Per-binary-column analysis (text-level categories)

In [ ]:
# Compare metrics split by each binary text-level column
label_cols = [
    "sexist",
    "irony",
    "humor",
    "implicit sexism",
    "stereotypes",
    "inequality",
    "discrimination",
    "objectification",
    "critique",
    "reported_sexism",
]

split_metrics_records = []
for col in label_cols:
    for _, row in data.iterrows():
        text_id = row["num_id"]
        group = row[col]
        for participant in part:
            if text_id not in tfds.get(participant, {}):
                continue
            if text_id not in annotations_sum:
                continue
            m = compute_metrics(tfds[participant][text_id], annotations_sum[text_id])
            if m is not None:
                split_metrics_records.append(
                    {
                        "column": col,
                        "group": int(group),
                        "participant": participant,
                        "text_id": text_id,
                        **m,
                    }
                )

split_df = pd.DataFrame(split_metrics_records)
print(f"Computed {len(split_df)} records across {split_df['column'].nunique()} columns")

In [ ]:
# Summary: mean metrics per column, per group
summary = (
    split_df.groupby(["column", "group"])
    .agg(
        spearman=("spearman", "mean"),
        cross_entropy=("cross_entropy", "mean"),
        js_div=("js_div", "mean"),
        n_texts=("text_id", "nunique"),
        n_pairs=("text_id", "count"),
    )
    .unstack(level="group")
    .round(2)
)

# Flatten multi-level columns
summary.columns = ["_".join(str(c) for c in col).strip("_") for col in summary.columns]
print("=== Metrics split by text-level binary categories ===")
print(summary.to_string())

In [ ]:
# Visualize the delta (positive - negative) for each category
deltas = []
for col in label_cols:
    sub = split_df[split_df["column"] == col]
    pos = sub[sub["group"] == 1]
    neg = sub[sub["group"] == 0]
    if len(pos) > 0 and len(neg) > 0:
        for metric in ["spearman", "cross_entropy", "js_div"]:
            deltas.append(
                {
                    "column": col,
                    "metric": metric,
                    "mean_positive": pos[metric].mean(),
                    "mean_negative": neg[metric].mean(),
                    "delta": pos[metric].mean() - neg[metric].mean(),
                }
            )

delta_df = pd.DataFrame(deltas)
print("Delta (positive group - negative group) per category:")
print(delta_df.pivot(index="column", columns="metric", values="delta").round(2).to_string())

In [ ]:
# Bar chart of deltas per category
pivot = delta_df.pivot(index="column", columns="metric", values="delta")

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, metric in zip(axes, ["spearman", "cross_entropy", "js_div"]):
    vals = pivot[metric].sort_values()
    colors = ["#2ca02c" if v > 0 else "#d62728" for v in vals]
    ax.barh(vals.index, vals, color=colors)
    ax.set_xlabel(f"Delta {metric}")
    ax.set_title(f"Delta {metric} (has label - no label)")
    ax.axvline(0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

# continues

In [ ]:
# tsv_file = "sexism2 Recording3.tsv"
tsv_file = "sexism4 Data export"
aoi_hit, calibration_df, participants, df = read_data(tsv_file)

In [ ]:
text_dfs, aoi_cols_dict = process_all_texts(df, aoi_hit)

In [ ]:
text_df = text_dfs["2"]
aoi_cols = aoi_cols_dict["2"]

In [ ]:
fixations = extract_fixations(text_df)
fixations

In [ ]:
regressions = detect_regressions(fixations)
metrics = compute_regression_metrics(fixations, regressions)
regressions

# Other

In [ ]:
# això té sentit si deixem totes les columnes d'AOI
if aoi_hit[0] in text_df.columns:
    aoi_data = text_df[aoi_cols].fillna(0)

    # Check for simultaneous hits
    hits_per_row = aoi_data.sum(axis=1)
    multi_hits = hits_per_row[hits_per_row > 1]

    text_df.loc[multi_hits.index].drop(set(aoi_cols) - set(aoi_cols[152:154]), axis=1)

In [ ]:
# df[~df.Event.isna()]

In [ ]:
# raw_df[~raw_df.Event.isna()]

In [ ]:
# df[["gaze_x", "gaze_y"]]

In [ ]:
# df[df[aoi_hit[0]] != 0]

# Pupil diameter

In [ ]:
# hits to int
df = z_score(df, "Pupil diameter filtered", "participant")

In [ ]:
text_hits = get_hits(df, aoi_hit, "participant", normalize=True)

In [ ]:
# todo: is there a "text" column?, else -> create it
# -> unique [text, participant] to check if they did all the texts.

In [ ]:
texts = {t.split(" - ")[1].rsplit("]", 1)[0] for t in aoi_hit}
# texts = [t.split("[")[1].split()[0] for t in aoi_hit] # old?
text = list(texts)[2]
print(text)
plt.scatter(text_hits[text]["hits"], text_hits[text]["z_pupil"])
plt.show()